# Group 8 Defect LLM: Tillicum Notebook Run

Use this notebook in Tillicum Open OnDemand Jupyter when `sbatch` setup is blocked. Run cells from top to bottom. The notebook uses the project `src/` directory directly, so it does not need `pip install -e .`.

## 1. Project Path

Set `PROJECT_ROOT` to the folder that contains `configs/`, `scripts/`, `src/`, and `requirements.txt`.

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    # Change this if your extracted folder has a different location.
    PROJECT_ROOT = Path("/gscratch/imt526a/group8/final")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python =", sys.executable)
print("Has configs:", (PROJECT_ROOT / "configs").exists())
print("Has src:", (PROJECT_ROOT / "src").exists())

## 2. Install Missing Packages Into User Site

This avoids writing into the read-only system `jupyter` environment. After this finishes, restart the notebook kernel if imports still fail.

In [ ]:
%pip install --user -r requirements.txt

## 3. Environment Check

In [ ]:
import importlib.util

for pkg in ["torch", "transformers", "datasets", "peft", "sklearn", "pytest", "yaml"]:
    print(f"{pkg:12s}", bool(importlib.util.find_spec(pkg)))

import torch
print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))

## 4. Unit Tests

In [ ]:
!python -m pytest -q

## 5. Prepare Data And Inspect Split Summary

This downloads/stages CodeXGLUE and writes `data/processed/.../split_summary.csv`.

In [ ]:
!python scripts/prepare_data.py --config configs/default.yaml

In [ ]:
import pandas as pd
pd.read_csv("data/processed/code_x_glue_cc_defect_detection/split_summary.csv")

## 6. Token-Length Profiling

In [ ]:
!python scripts/profile_tokens.py --config configs/default.yaml --split train

## 7. CPU Baselines In Notebook

Run this before LLM training. It can take several minutes.

In [ ]:
!python scripts/run_baselines.py --config configs/default.yaml --eval-split validation

In [ ]:
pd.read_csv("outputs/baselines/baseline_validation_summary.csv")

## 8. Optional GPU Smoke Test

Only run this if the notebook session has a GPU. This trains the 2,000-example bf16 LoRA smoke config.

In [ ]:
import torch
assert torch.cuda.is_available(), "This notebook kernel has no GPU. Start a GPU Jupyter session before running this cell."
os.environ["WANDB_PROJECT"] = "group8_defect_detection"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!python scripts/train_sft.py --config configs/smoke_lora_512.yaml

## 9. Optional Smoke Validation Eval

In [ ]:
assert torch.cuda.is_available(), "This notebook kernel has no GPU."
!python scripts/evaluate_llm.py --config configs/smoke_lora_512.yaml --split validation --adapter outputs/smoke_lora_512/final_adapter

## 10. Full Main Run

Run this only after the smoke test works. This may take hours. Keep the browser/session alive or use SLURM once partition settings are fixed.

In [ ]:
# assert torch.cuda.is_available(), "This notebook kernel has no GPU."
# !python scripts/train_sft.py --config configs/qlora_512.yaml
# !python scripts/evaluate_llm.py --config configs/qlora_512.yaml --split validation --adapter outputs/qwen25_7b_qlora_512/final_adapter